# 23-16 · Cofnij i konflikt na przywróceniu

Praktyka do sekcji [„Cofnij ostatnią operację” `safesort`.

## Reproducible local environment

```bash
git clone https://github.com/Cartesian-School/safesort.git
cd safesort
python3.14 -m venv .venv
source .venv/bin/activate
# Windows PowerShell: .venv\Scripts\Activate.ps1
python -m pip install -U pip
python -m pip install -e ".[dev]"
python -m pip install jupyter ipykernel
python -m ipykernel install --user --name safesort-py314 --display-name "SafeSort Python 3.14"
jupyter lab
```

Select the **SafeSort Python 3.14** kernel. The diagnostic cell below must
point into this `.venv` and the cloned `src/safesort` tree.

In [ ]:
import sys
import safesort

print(sys.executable)
print(safesort.__file__)

## Cel

Stosuj plan, anuluj go tym `safesort.manifest.undo()` i upewnić się, że w przypadku konfliktu (na początkowym miejscu już coś jest) anulowanie odmawia nadpisania.

## Example

In [ ]:
import tempfile
from pathlib import Path

from safesort.config import Config
from safesort.scanner import scan
from safesort.planner import build_plan
from safesort.executor import apply_plan
from safesort.manifest import write_manifest, read_manifest, find_latest_manifest, undo

tmpdir = tempfile.TemporaryDirectory()
koren = Path(tmpdir.name)

(koren / "otchet.pdf").write_text("отчёт", encoding="utf-8")
(koren / "zametka.txt").write_text("заметка", encoding="utf-8")

nastrojki = Config()
fajly = scan(koren, nastrojki)
plan = build_plan(fajly, koren, nastrojki)
rezultaty = apply_plan(plan)
_manifest_obj, put_k_manifestu = write_manifest(koren, rezultaty)

print("Перемещено файлов:", sum(1 for r in rezultaty if r.completed))
print("Манифест записан в:", put_k_manifestu)

## Sprawdzenie wyniku — pliki faktycznie zostały przeniesione

In [ ]:
assert not (koren / "otchet.pdf").exists()
assert not (koren / "zametka.txt").exists()
assert (koren / "Sorted" / "documents" / "otchet.pdf").exists()
assert put_k_manifestu.exists()
print("Верно: оба файла оказались в Sorted/documents/, манифест записан на диск.")

## Eksperyment – undo przywraca pliki

In [ ]:
najdennyj_manifest = find_latest_manifest(koren)
manifest_dlya_otmeny = read_manifest(najdennyj_manifest)

rezultat_otmeny = undo(manifest_dlya_otmeny)

assert (koren / "otchet.pdf").exists()
assert (koren / "zametka.txt").exists()
assert rezultat_otmeny.conflicts == ()
print("Верно: оба файла вернулись на исходное место, конфликтов не было.")

## Starter

Wypełnij zaznaczone miejsce. Niezmieniony starter nie przechodzi tests.

In [ ]:
def proverit_konflikt_undo(root: Path):
    # TODO: apply one file, write a replacement at the source, then call undo.
    raise NotImplementedError


## Task

Napisz skrypt, w którym nowy plik pojawi się na oryginalnej ścieżce po apply i undo zgłasza jeden konflikt i nie nadpisuje go.

## Tests

Run After task cell: jest podstawowy przykład i przynajmniej jeden skrajny przypadek.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    test_root = Path(tmp)
    konflikt, content = proverit_konflikt_undo(test_root)
assert konflikt is True
assert content == "новый файл"
print("Tests passed")

## Hint

Oszczędź manifest później `apply_plan`, następnie utwórz nowy source do `undo`.

## Solution

Pokaż rozwiązanie po własnej próbie</summary>

```python
def proverit_konflikt_undo(root: Path):
    source = root / "otchet.pdf"
    source.write_text("оригинал", encoding="utf-8")
    config = Config()
    plan = build_plan(scan(root, config), root, config)
    moves = apply_plan(plan)
    manifest, _ = write_manifest(root, moves)

    source.write_text("новый файл", encoding="utf-8")
    result = undo(manifest)
    return len(result.conflicts) == 1, source.read_text(encoding="utf-8")
```

</details>